# 🤖💻 **Deploy a Text Generation Model with Streamlit + MLflow**

**Time Estimate:** 45–60 minutes

## 📋 **Overview**

In this programming activity, you will deploy a pre-trained text generation model using MLflow for model management and Streamlit for building an intuitive web interface. Understanding how to integrate AI into accessible applications is crucial for AI specialists who aim to build user-friendly tools and solutions.

This activity demonstrates the practical process of deploying AI models, a skill highly relevant for roles in AI development and machine learning engineering.

## 🎯 **Learning Outcomes**

By the end of this lab, you will be able to:

- Set up and manage models with MLflow
- Develop an interactive web interface using Streamlit
- Deploy an AI application locally using Jupyter and Streamlit

In [2]:
!pip uninstall -y tensorflow tensorflow-cpu keras scipy numpy >/dev/null 2>&1

!pip install -q \
    "numpy==1.26.4" \
    "scipy==1.13.1" \
    "transformers==4.44.2" \
    "mlflow"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires requests>=2.32.2, but you have requests 2.31.0 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip


## Task 1: Wrap the Model with MLflow [20 minutes]

In [1]:
import mlflow
import mlflow.pyfunc
from transformers import pipeline
import pandas as pd
import warnings# imports

warnings.filterwarnings("ignore", category=UserWarning)

Configure and wrap a pre-trained model for deployment.

In [2]:
import mlflow
import mlflow.pyfunc
from transformers import pipeline
import pandas as pd
import warnings

warnings.filterwarnings("ignore", category=UserWarning)


# Load text generation model
generator = pipeline(
    "text-generation",
    model="gpt2"
)


# Create MLflow wrapper
class TextGenerationModel(mlflow.pyfunc.PythonModel):

    def __init__(self, generator):
        self.generator = generator

    def predict(self, context, model_input, params=None):

        # Handle pandas DataFrame
        if isinstance(model_input, pd.DataFrame):
            prompts = model_input.iloc[:, 0].tolist()

        # Handle pandas Series
        elif isinstance(model_input, pd.Series):
            prompts = model_input.tolist()

        # Handle list
        elif isinstance(model_input, list):
            prompts = model_input

        # Handle single string
        elif isinstance(model_input, str):
            prompts = [model_input]

        else:
            raise ValueError("Unsupported input format")

        results = []

        for prompt in prompts:
            output = self.generator(
                prompt,
                max_new_tokens=50,
                num_return_sequences=1,
                do_sample=True
            )

            results.append(output[0]["generated_text"])

        return results


# Save the model using MLflow
model_path = "text_generation_model"

mlflow.pyfunc.save_model(
    path=model_path,
    python_model=TextGenerationModel(generator)
)

print("Model saved successfully at:", model_path)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.
/usr/local/lib/python3.11/dist-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


Model saved successfully at: text_generation_model


🔍 **Practice**

Consider how MLflow helps manage model lifecycle and versioning. Think about:

- How to create a custom model wrapper that handles different input formats.
- The importance of proper predict method implementation for MLflow compatibility.

✅ **Success Checklist**

- Model is successfully initialized and wrapped
- MLflow model wrapper handles input correctly
- Model is saved to the specified path

💡 **Key Points**

- MLflow model wrappers need proper predict method signatures
- Handle different input formats (pandas Series/DataFrame, lists)
- Text generation parameters should be configurable

❗ **Common Mistakes to Avoid**

- Not handling pandas input formats in the predict method
- Forgetting to extract generated text from Hugging Face output format
- Not setting appropriate generation parameters

## Task 2: Create Streamlit Interface [20 minutes]
Develop a user-friendly interface for text input and output.
1. Create Streamlit components for user input
2. Load the MLflow model within the interface
3. Handle text generation and display results
4. Add interactive elements for better user experience

In [3]:
import streamlit as st
import mlflow.pyfunc


st.title("Text Generation App")

st.write("Generate text using an MLflow-managed GPT-2 model.")


# User input
prompt = st.text_area(
    "Enter your prompt:",
    placeholder="Write something to start the generation..."
)


# Generate button
if st.button("Generate Text"):

    if prompt.strip() == "":
        st.warning("Please enter a prompt.")

    else:

        # Load MLflow model
        model = mlflow.pyfunc.load_model(
            "text_generation_model"
        )

        # Generate text
        result = model.predict([prompt])

        # Display result
        st.subheader("Generated Text")
        st.write(result[0])

2026-09-11 11:02:00.176 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-11 11:02:00.292 
  command:

    streamlit run /usr/local/lib/python3.11/dist-packages/ipykernel_launcher.py [ARGUMENTS]
2026-09-11 11:02:00.293 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-11 11:02:00.293 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-11 11:02:00.294 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-11 11:02:00.294 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-11 11:02:00.295 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-11 11:02:00.295 Thread 'MainThread': missi

🔍 **Practice**

Think about how to create an effective user interface. Consider:

- What Streamlit components are best for text input and display.
- How to handle model loading and text generation within the interface.

✅ **Success Checklist**

- Streamlit interface displays properly in notebook
- Text input and generation button work correctly
- Generated text displays in the interface

💡 **Key Points**

- Streamlit can run directly in Jupyter notebook cells
- ScriptRunContext warnings are normal in notebook environment
- Model loading should happen within the button click handler

❗ **Common Mistakes to Avoid**

- Trying to use session state without proper Streamlit context
- Not handling the model input format correctly
- Ignoring the ScriptRunContext warnings (they're expected)

## Task 3: Test the Deployment [15 minutes]
Launch and test your application using Streamlit.
1. Use shell command to run Streamlit properly
2. Test the text generation functionality
3. Verify the complete workflow works
4. Document the deployment URLs and access methods

In [8]:
!ls

app.py	deploy.ipynb  mlruns  text_generation_model


In [ ]:
!streamlit run app.py




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.18.0.10:8501
  External URL: http://18.209.53.92:8501



🔍 **Practice**

Testing deployment requires understanding the environment. Consider:

- How to properly launch Streamlit from the notebook environment.
- What URLs and ports will be available for testing.

✅ **Success Checklist**

- Streamlit application launches successfully
- Local and network URLs are displayed
- Text generation works in the web interface

💡 **Key Points**

- Shell commands with `!` can launch Streamlit from notebooks
- Multiple URL options (local, network, external) may be available
- Deployment testing validates the complete pipeline

❗ **Common Mistakes to Avoid**

- Not using the correct launcher path for the environment
- Expecting immediate browser opening (manual navigation required)
- Not testing the full text generation workflow

🚀 **Next Steps**

Explore more on optimizing model inference pipelines for improved performance, and learn about deploying scalable AI services using cloud-based solutions like AWS or GCP. Consider containerizing your application with Docker for more robust deployment options.

## 💻 Exemplar Solution

<details>    
<summary><strong>Click HERE to see an exemplar solution</strong></summary>

### Task 1 Solution
    
```python
# Load your pre-trained model
model_name = "distilgpt2"
generator = pipeline("text-generation", model=model_name)

print(f"Loaded model: {model_name}")

# Create MLflow model wrapper
class TextGenWrapper(mlflow.pyfunc.PythonModel):
    def __init__(self, generator, max_new_tokens=50):
        self.generator = generator
        self.max_new_tokens = max_new_tokens

    def predict(self, context, model_input):
        # Handle pandas Series/DataFrame or list-like input
        if isinstance(model_input, (pd.Series, pd.DataFrame)):
            prompts = model_input.squeeze().tolist()
        else:
            prompts = model_input
        
        # Generate text
        outputs = self.generator(
            prompts,
            max_new_tokens=self.max_new_tokens,
            do_sample=True,
            temperature=0.7
        )
        
        # Extract generated text from Hugging Face output format
        return [out[0]["generated_text"] for out in outputs]

# Create the model wrapper
pyfunc_model = TextGenWrapper(generator)

# Save model with MLflow
mlflow.pyfunc.save_model(
    path="mlflow_model",
    python_model=pyfunc_model
)

print("Model successfully saved to 'mlflow_model'")
```

### Task 2 Solution
    
```python
import streamlit as st

# Set up the Streamlit interface
st.title("🤖 LLM-powered Text Generation")
st.markdown("Generate creative text using our pre-trained language model!")

# User input
user_input = st.text_area(
    "Enter your prompt:",
    placeholder="Once upon a time in a distant galaxy...",
    height=100
)

# Generation settings
col1, col2 = st.columns(2)
with col1:
    max_tokens = st.slider("Max new tokens:", 10, 100, 50)
with col2:
    temperature = st.slider("Temperature:", 0.1, 1.0, 0.7)

# Generate button
if st.button("🚀 Generate Text"):
    if user_input.strip():
        with st.spinner("Generating text..."):
            try:
                # Load the MLflow model
                model = mlflow.pyfunc.load_model("mlflow_model")
                
                # Generate response
                result = model.predict([user_input])
                
                # Display result
                st.subheader("📝 Generated Text:")
                st.write(result[0])
                
            except Exception as e:
                st.error(f"Error generating text: {e}")
    else:
        st.warning("⚠️ Please enter a prompt to generate text!")

# Note about warnings
st.info("ℹ️ ScriptRunContext warnings in the console are normal when running Streamlit in notebooks.")
```

### Task 3 Solution

```python
# Launch Streamlit app using shell command
# This will start the Streamlit server and display available URLs

print("🚀 Launching Streamlit application...")
print("The app will display Local URL, Network URL, and possibly External URL")
print("Click on any of these URLs to access your deployed application")
print("\n⚠️ Note: The app will run until you stop the cell execution")

# Run Streamlit (this will use the code from Task 2)
!streamlit run /usr/local/lib/python3.11/dist-packages/colab_kernel_launcher.py
```

</details>